# Air Quality Prediction - Initial EDA
**Proyek**: MLOps-AirQualityPrediction
**Tujuan**: Eksplorasi awal data PM2.5 dan fitur meteorologi dari OpenWeatherMap API
**Kota**: Jakarta, Surabaya, Bandung, Medan, Semarang


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests, os

# Konfigurasi
API_KEY = os.getenv('OPENWEATHER_API_KEY', 'YOUR_KEY_HERE')
CITIES = [
    {'name': 'Jakarta',  'lat': -6.2088, 'lon': 106.8456},
    {'name': 'Surabaya', 'lat': -7.2575, 'lon': 112.7521},
    {'name': 'Bandung',  'lat': -6.9175, 'lon': 107.6191},
]
print(f'API Key terkonfigurasi: {bool(API_KEY)}')
print(f'Jumlah kota: {len(CITIES)}')

In [ ]:
# Test koneksi API - ambil data 1 kota
city = CITIES[0]
url = f'http://api.openweathermap.org/data/2.5/air_pollution'
params = {'lat': city['lat'], 'lon': city['lon'], 'appid': API_KEY}
response = requests.get(url, params=params, timeout=10)

if response.status_code == 200:
    data = response.json()
    print(f'API berhasil! Data untuk {city["name"]}:')
    print(f'AQI: {data["list"][0]["main"]["aqi"]}')
    print(f'PM2.5: {data["list"][0]["components"]["pm2_5"]} ug/m3')
else:
    print(f'Error: {response.status_code} - cek API key kamu')

In [ ]:
# Ambil data semua kota dan buat DataFrame
records = []
for city in CITIES:
    r = requests.get(url, params={'lat': city['lat'], 'lon': city['lon'], 'appid': API_KEY})
    if r.status_code == 200:
        d = r.json()['list'][0]
        rec = {
            'city': city['name'],
            'aqi': d['main']['aqi'],
            **d['components']  # pm2_5, pm10, no2, o3, co, so2
        }
        records.append(rec)

df = pd.DataFrame(records)
print(df.shape)
df

In [ ]:
# Visualisasi distribusi PM2.5 per kota
fig, ax = plt.subplots(figsize=(10, 5))
df.set_index('city')[['pm2_5', 'pm10', 'no2']].plot(kind='bar', ax=ax)
ax.set_title('Konsentrasi Polutan per Kota (snapshot)')
ax.set_ylabel('ug/m3')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/01_pollutants_by_city.png', dpi=150)
plt.show()

## Temuan Awal
- API berhasil mengembalikan data PM2.5, PM10, NO2, O3, CO, SO2
- Terdapat variasi konsentrasi polutan yang signifikan antar kota
- Data update setiap jam — perlu ingestion otomatis (APScheduler)
- Label AQI kategori perlu dihitung dari PM2.5 menggunakan standar BMKG
